In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("/home1/smaruj/ledidi_akita/"))
from utils.df_utils import load_parameter_results

In [2]:
_PROJ     = "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita"
RESULTS_DIR = f"{_PROJ}/optimizations/boundaries"

FOLDS   = [0, 1, 2, 3]
TAUS = [0.01, 0.1, 1.0, 10.0]

In [3]:
# Tau sweep
df_tau = load_parameter_results(RESULTS_DIR, "tau", TAUS, folds=FOLDS)

Total windows loaded: 696


In [4]:
df_tau["no_edits"] = df_tau["n_edits"] == 0

no_edits_summary = (
    df_tau.groupby("tau")["no_edits"]
    .agg(n_total="count", n_no_edits="sum")
    .assign(pct_no_edits=lambda x: 100 * x["n_no_edits"] / x["n_total"])
)
print(no_edits_summary.to_string())

       n_total  n_no_edits  pct_no_edits
tau                                     
0.01       174           0           0.0
0.10       174           0           0.0
1.00       174           0           0.0
10.00      174           0           0.0


In [5]:
df_tau["insul_diff"] = df_tau["insul_score_edited"] - df_tau["insul_score_orig"]
df_tau["no_improvement"] = df_tau["insul_diff"] >= 0

no_improvement_summary = (
    df_tau.groupby("tau")["no_improvement"]
    .agg(n_total="count", n_no_improvement="sum")
    .assign(pct_no_improvement=lambda x: 100 * x["n_no_improvement"] / x["n_total"])
)
print(no_improvement_summary.to_string())

       n_total  n_no_improvement  pct_no_improvement
tau                                                 
0.01       174                 6            3.448276
0.10       174                 0            0.000000
1.00       174                 0            0.000000
10.00      174                 0            0.000000


In [6]:
success_rate = (
    df_tau.assign(success=~df_tau["no_edits"] & ~df_tau["no_improvement"])
    .groupby("tau")["success"]
    .agg(n_total="count", n_success="sum")
    .assign(pct_success=lambda x: 100 * x["n_success"] / x["n_total"])
)
print(success_rate.to_string())

       n_total  n_success  pct_success
tau                                   
0.01       174        168    96.551724
0.10       174        174   100.000000
1.00       174        174   100.000000
10.00      174        174   100.000000


In [7]:
summary = (
    df_tau.groupby("tau")
    .agg(
        n_success        = ("n_edits",          "count"),
        mean_n_edits     = ("n_edits",          "mean"),
        mean_insul_diff  = ("insul_diff",        "mean"),
        mean_CTCFs       = ("CTCFs_num",         "mean"),
    )
    .round(3)
)
print(summary.to_string())

       n_success  mean_n_edits  mean_insul_diff  mean_CTCFs
tau                                                        
0.01         174        47.270           -0.007       0.690
0.10         174       360.195           -0.218       4.040
1.00         174       792.098           -0.314       7.121
10.00        174      1079.172           -0.319       7.626
